# Validation Scientifique et Benchmarking

Dans les notebooks précédents, nous avons construit un système complexe. Mais est-il performant ?

Ce notebook a pour objectif de **quantifier la qualité** des réponses via deux approches :
1.  **Test "Ablation" (Baseline)** : Interroger le LLM *sans* contexte pour prouver qu'il ne connaît pas autant d'informations que contenues dans les documents.
2.  **Évaluation Quantitative** : Calculer un score de précision (Accuracy/Recall) sur un jeu de données de test (`golden_dataset.json`).


---

## Partie 1 : Test "Ablation" (Baseline)

## 1. Configuration de la Baseline (Modèle "Nu")
Dans cette cellule, nous chargeons le modèle uniquement avec sa **mémoire paramétrique** (ce qu'il a appris pendant son entraînement chez Meta), sans aucune connexion à notre base vectorielle FAISS.

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# SCRIPT DE TEST : BASELINE (LLM SANS RAG)
# Ce script sert à démontrer les limites du modèle "nu".

# CONFIGURATION
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

print(f"[INIT] Loading LLM model: {MODEL_NAME}")

/home/aminedesel/Desktop/projets/NLP_Projet/NLPenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INIT] Loading LLM model: meta-llama/Llama-3.2-1B-Instruct


## 2. Optimisation et Détection du Matériel

Tout comme pour le pipeline RAG principal, nous configurons ici l'environnement d'exécution pour le test de la Baseline.

In [2]:
# 1. Configuration Matérielle
# On vérifie si un GPU est disponible pour accélérer l'inférence.
if torch.cuda.is_available() or torch.backends.mps.is_available():
    llm_dtype = torch.bfloat16 # Mode rapide (16 bits)
    print("-> Mode: GPU Acceleration")
else:
    llm_dtype = torch.float32  # Mode standard (32 bits)
    print("-> Mode: CPU")

-> Mode: GPU Acceleration


## 3. Chargement du Modèle

Nous chargeons ici les **poids bruts** du modèle depuis HuggingFace.



Pour que ce test soit scientifiquement valide (comparaison "toutes choses égales par ailleurs"), nous utilisons **exactement la même configuration** que dans le notebook RAG :
1.  **Le Tokenizer** : L'interprète qui traduit le texte en vecteurs numériques.
2.  **Le Modèle** : Le réseau de neurones (`AutoModelForCausalLM`) qui prédit la suite des mots.

In [4]:
# 2. Chargement du Tokenizer et du Modèle
# Le Tokenizer transforme le texte en suite de nombres.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Le Modèle est chargé avec gestion automatique de la mémoire (device_map="auto")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=llm_dtype,
    device_map="auto", 
)

## 4. Fonction d'Inférence Brute ("Raw Query")

Cette fonction est l'interface directe avec le LLM. Elle envoie une question et récupère la réponse brute, sans artifice.



**Le Pipeline de Génération NLP :**

Le code suit les 4 étapes classiques du Traitement du Langage Naturel :

1.  **Tokenization (`tokenizer`)** : La phrase est découpée en tokens et convertie en une suite de nombres (Tenseurs). C'est la seule langue que le modèle comprend.
2.  **Device Transfer (`.to(device)`)** : On déplace ces nombres vers le processeur qui contient le modèle (GPU ou CPU).
3.  **Inférence (`model.generate`)** :
    * Le modèle prédit la suite des tokens un par un.
    * **Note importante :** Nous utilisons `do_sample=False`. Cela force une **génération déterministe** (Greedy Search). Pour une même question, le modèle donnera toujours exactement la même réponse. C'est essentiel pour un benchmark reproductible.
4.  **Decoding (`tokenizer.decode`)** : On traduit les nombres prédits en texte lisible pour l'humain.

In [5]:
def call_llm_raw(text: str) -> str:
    # A. Tokenization
    # On convertit la question en tenseurs PyTorch ('pt')
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    )
    
    # B. Transfert sur GPU si disponible
    # Il faut que les données soient au même endroit que le modèle
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # C. Génération
    with torch.no_grad(): # On désactive le calcul de gradients (économie mémoire)
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,      # Longueur max de la réponse
            do_sample=False,         # False = Réponse déterministe (toujours la même)
            pad_token_id=tokenizer.eos_token_id,
        )

    # D. Décodage
    # On retire la question (input_ids) pour ne garder que la réponse générée
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True, # On enlève les balises techniques (<EOS>, <BOS>)
        clean_up_tokenization_spaces=True,
    ).strip()

    if not answer:
        answer = "The model did not generate a valid answer."

    return answer

## 5. Démo Interactive

Cette fonction lance une boucle de chat interactive pour tester le modèle "à nu".

**Objectif du test :**
Nous cherchons à mettre en évidence les **hallucinations** ou le manque de connaissances. 

In [6]:
def llm_base():
    print("=== TEST : PURE LLM (NO RAG) ===")
    print("Ce script interroge le modèle sans accès à vos documents.")
    print("Utilisez-le pour mettre en évidence les hallucinations.")
    print("Tapez 'q' pour quitter.")

    while True:
        query = input("\nVotre question : ").strip()
        if query.lower() in {"q", "quit", "exit"}:
            print("Bye ~")
            break

        # Appel direct (Raw)
        answer = call_llm_raw(query)

        print("\n===== Réponse du Modèle (Sans contexte) =====")
        print(answer)
        print("=============================================")


llm_base()

=== TEST : PURE LLM (NO RAG) ===
Ce script interroge le modèle sans accès à vos documents.
Utilisez-le pour mettre en évidence les hallucinations.
Tapez 'q' pour quitter.


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



===== Réponse du Modèle (Sans contexte) =====
If left untreated, HIV can lead to a range of serious health problems, including:
1. **AIDS**: HIV can cause AIDS, which is a life-threatening condition that can lead to serious health problems, such as:
	* **Kaposi's sarcoma**: a type of skin cancer that can cause lesions, ulcers, and scarring.
	* **Lymphoma**: a type of cancer that can affect the immune system and cause symptoms such as fever, fatigue, and swollen lymph nodes.
	* **Pneumocystis pneumonia**: a type of lung infection that can cause symptoms such as coughing, fever, and difficulty breathing.
	* **Meningitis**: a type of infection that can cause symptoms such as fever, headache, and stiff neck.
	* **HIV-related cancers**: such as cervical, anal, and penile cancers.
2. **Neurological problems**: HIV can cause neurological problems, such as:
	* **Neuropathy**: a condition that can cause numbness, tingling, and pain in the hands and feet.
	* **Dementia**: a condition that can c

# Partie 2 : Évaluation Quantitative (Benchmark)

Après avoir constaté manuellement le comportement du modèle, nous passons à une évaluation **automatisée et chiffrée**.

L'objectif est de calculer des métriques précises sur un jeu de données de test ("Golden Dataset").



### 1. Imports et Outillage
Nous chargeons ici l'ensemble des bibliothèques nécessaires pour :
1.  **Reconstruire le Pipeline RAG** (FAISS, BM25, Cross-Encoder) afin de générer les réponses.
2.  **Manipuler les Données** (`json`, `pandas`) pour gérer le dataset de test.
3.  **Suivre la progression** (`tqdm`) car l'évaluation sur beaucoup de questions peut prendre du temps.

In [14]:
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

## 2. Configuration de l'Environnement de Test

Cette étape prépare l'environnement pour l'évaluation automatisée.



**Points clés de la configuration :**

1.  **Résolution Dynamique (`BASE_DIR`)** :
    * Le script remonte automatiquement l'arborescence des dossiers jusqu'à trouver le répertoire `data`.

2.  **Le Golden Dataset (`DATASET_PATH`)** :
    * Il contient la "Vérité Terrain" (Ground Truth).

3.  **Les Seuils (`THRESHOLDS`)** :
    * Nous fixons les mêmes seuils de décision que dans l'application principale pour garantir que le test reflète la réalité de la production.

In [15]:
# 1. CONFIGURATION ET CHEMINS

# Détection automatique de la racine du projet
BASE_DIR = Path.cwd()
while not (BASE_DIR / "data").exists():
    if BASE_DIR == BASE_DIR.parent:
        raise FileNotFoundError("Dossier 'data' introuvable.")
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "data"
PROC_DIR = DATA_DIR / "processed"
INDEX_DIR = DATA_DIR / "index"
DATASET_PATH = DATA_DIR / "golden_dataset.json"

# Seuils de décision
THRESHOLD_STRICT = 0.0      # Pour V4 (Logits du Reranker)
THRESHOLD_SIMPLE = 0.45     # Pour V1 (Similarité Cosinus FAISS)

## 3. Reconstitution du Moteur de Recherche

Pour évaluer notre RAG, nous devons recharger **l'intégralité de la chaîne de recherche**.

**Les composants du banc d'essai :**
1.  **Corpus & FAISS** : La mémoire vectorielle brute.
2.  **Embedding Model** : Pour encoder les questions du test.
3.  **BM25** : Pour tester la recherche par mots-clés.
4.  **Cross-Encoder** : Pour valider l'étape de filtrage final.

In [16]:
# 2. CHARGEMENT DES RESSOURCES

def load_resources():
    print("--- Chargement des ressources RAG ---")
    
    # 1. Corpus (Textes)
    corpus_path = PROC_DIR / "docs_corpus.csv"
    if not corpus_path.exists():
        raise FileNotFoundError(f"Fichier introuvable : {corpus_path}")
    df = pd.read_csv(corpus_path)
    
    # 2. Index FAISS (Vecteurs)
    faiss_path = INDEX_DIR / "corpus.index"
    index = faiss.read_index(str(faiss_path))

    # 3. Modèle d'Embedding (Traducteur Texte -> Vecteur)
    model_name_path = INDEX_DIR / "embedding_model.txt"
    model_name = model_name_path.read_text(encoding="utf-8").strip()
    embed_model = SentenceTransformer(model_name)
    
    # 4. Reranker (Juge de pertinence)
    reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

    # 5. Index BM25 (Mots-clés)
    tokenized_corpus = [str(doc).lower().split(" ") for doc in df['text']]
    bm25 = BM25Okapi(tokenized_corpus)

    return df, index, embed_model, reranker, bm25


## 4. Briques de Recherche (Réplication de la Logique)

Pour que l'évaluation soit valide, nous devons utiliser **exactement les mêmes algorithmes** de recherche que ceux définis dans le notebook principal.



Nous redéfinissons ici les 4 composants clés du pipeline "Hybrid RAG" :

1.  **Dense Retrieval (`retrieve_faiss`)** : La recherche sémantique vectorielle.
2.  **Sparse Retrieval (`retrieve_bm25`)** : La recherche par mots-clés (pour capter le jargon spécifique).
3.  **Fusion (`reciprocal_rank_fusion`)** : L'algorithme qui marie le meilleur des deux mondes.
4.  **Reranking (`rerank_contexts`)** : Le filtre de précision ultime via Cross-Encoder

In [17]:
# 3. BRIQUES DE RECHERCHE (ATOMIQUES)

def retrieve_faiss(query, df, index, embed_model, top_k=10):
    """Recherche Vectorielle (Sémantique)"""
    query_emb = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = index.search(query_emb, top_k)
    contexts = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0 or idx >= len(df): continue
        row = df.iloc[idx]
        contexts.append({
            "doc_id": row.get("doc_id", idx),
            "score": float(score),
            "text": str(row["text"])
        })
    return contexts

def retrieve_bm25(query, df, bm25, top_k=10):
    """Recherche Lexicale (Mots-clés exacts)"""
    tokenized_query = query.lower().split(" ")
    top_docs = bm25.get_top_n(tokenized_query, df['text'].tolist(), n=top_k)
    contexts = []
    for text in top_docs:
        # Note : Retrouver la ligne via le texte est lent mais ok pour l'évaluation
        matches = df[df['text'] == text]
        if not matches.empty:
            row = matches.iloc[0]
            contexts.append({
                "doc_id": row.get("doc_id"),
                "score": 0.0, 
                "text": text
            })
    return contexts

def reciprocal_rank_fusion(list_a, list_b, k=60):
    """Fusionne les résultats FAISS et BM25"""
    scores_map = {}
    def add_to_map(results_list):
        for rank, doc in enumerate(results_list):
            key = doc['text'] 
            if key not in scores_map:
                scores_map[key] = {"doc": doc, "score": 0.0}
            scores_map[key]["score"] += 1 / (k + rank + 1)
    add_to_map(list_a)
    add_to_map(list_b)
    fused_sorted = sorted(scores_map.values(), key=lambda x: x['score'], reverse=True)
    return [item['doc'] for item in fused_sorted]

def rerank_contexts(query, contexts, reranker, top_k=5):
    """Réordonne les résultats avec le Cross-Encoder"""
    if not contexts: return []
    pairs = [[query, doc['text']] for doc in contexts]
    scores = reranker.predict(pairs)
    for i, doc in enumerate(contexts):
        doc['score'] = float(scores[i])
    return sorted(contexts, key=lambda x: x['score'], reverse=True)[:top_k]

## 5. Définition des Pipelines à Évaluer

Nous définissons ici les deux architectures à comparer.

### Pipeline V1 : "Le Naïf" (Baseline)
* **Technique** : Simple recherche vectorielle (FAISS) basée sur la similarité cosinus.
* **Hypothèse** : Rapide mais risque de manquer de précision sur des termes techniques exacts ou des questions complexes.

### Pipeline V4 : "L'Expert" (Hybrid RAG)
* **Technique** : Combinaison de FAISS (Sens) + BM25 (Mots-clés) + Reranking (Cross-Encoder).
* **Hypothèse** : Beaucoup plus robuste. Le Reranker devrait filtrer le bruit et remonter les documents les plus pertinents en tête de liste.

In [18]:
# 4. PIPELINES À COMPARER

def pipeline_v1_simple(query, df, index, embed_model, top_k=5):
    """Pipeline V1 : Recherche FAISS simple"""
    return retrieve_faiss(query, df, index, embed_model, top_k)

def pipeline_v4_hybrid(query, df, index, embed_model, bm25, reranker, top_k=5):
    """Pipeline V4 : Hybride (FAISS+BM25) -> Fusion -> Reranking"""
    res_faiss = retrieve_faiss(query, df, index, embed_model, top_k=15)
    res_bm25 = retrieve_bm25(query, df, bm25, top_k=15)
    fused_docs = reciprocal_rank_fusion(res_faiss, res_bm25)
    return rerank_contexts(query, fused_docs, reranker, top_k=top_k)

## 6. Métrique de Réussite : Le Hit Rate

La méthode la plus objective est de vérifier si le **document source attendu** (Golden Document) apparaît bien dans les résultats.



**La Logique de Notation (`calculate_success`) :**

* **Entrée** :
    * `retrieved_docs` : La liste des 5 documents trouvés par le RAG.
    * `expected_ids` : L'ID officiel du document qui contient la réponse (défini dans le *Golden Dataset*).
* **Critère de Succès (Hit)** :
    * Si l'ID attendu est présent dans le Top-5 $\rightarrow$ **Succès (1)**.
    * Sinon $\rightarrow$ **Échec (0)**.

In [19]:
# 5. CALCUL DES MÉTRIQUES

def calculate_success(retrieved_docs, expected_ids):
    """Renvoie True si au moins un ID de document attendu est trouvé"""
    if not expected_ids or not retrieved_docs:
        return False
    found_ids = [d.get('doc_id') for d in retrieved_docs]
    return any(e_id in found_ids for e_id in expected_ids)

## 7. Le Grand Duel : Exécution du Benchmark

Ce script parcourt chaque question du `golden_dataset.json` et lance les deux pipelines en parallèle.



### Logique de Notation (Scoring) :

Le script distingue deux cas de figure cruciaux pour un système robuste :

1.  **Questions Standards (`type="standard"`)** :
    * *Objectif :* Retrouver le bon document.
    * *Critère :* L'ID du document attendu (`expected_doc_ids`) est-il dans le Top-5 remonté ?
    * *Métrique :* **Hit Rate** (Taux de réussite).

2.  **Questions Pièges (`type="trap"`)** :
    * *Objectif :* Ne pas se faire avoir ! Le système doit dire "Je ne sais pas".
    * *Critère :* Le score de pertinence est-il **inférieur au seuil** de sécurité ?
    * *Métrique :* **Rejection Rate** (Taux de rejet correct).

In [20]:
# 6. EXÉCUTION DU BENCHMARK COMPARATIF

print(f"\n=== BENCHMARK : V1 (Simple) vs V4 (Hybride) ===")

if not DATASET_PATH.exists():
    print(f"[ERREUR] Dataset introuvable : {DATASET_PATH}")
else:
    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        dataset = json.load(f)
    print(f"[INFO] {len(dataset)} questions chargées.")

    # Chargement du moteur
    df, index, embed_model, reranker, bm25 = load_resources()

    # Compteurs de scores pour les deux versions
    scores = {
        "v1": {"correct": 0, "traps_caught": 0, "total_scorable": 0, "total_traps": 0},
        "v4": {"correct": 0, "traps_caught": 0, "total_scorable": 0, "total_traps": 0}
    }

    print("\n--- Lancement des tests (Comparaison V1 vs V4) ---")

    for item in tqdm(dataset):
        query = item["question"]
        q_type = item["type"]
        expected_ids = item.get("expected_doc_ids", [])

        # EXÉCUTION V1 
        docs_v1 = pipeline_v1_simple(query, df, index, embed_model, top_k=5)
        
        # EXÉCUTION V4
        docs_v4 = pipeline_v4_hybrid(query, df, index, embed_model, bm25, reranker, top_k=5)

        # ÉVALUATION
        if q_type == "trap":
            # LOGIQUE PIÈGE : Succès si aucun doc renvoyé OU score très bas
            scores["v1"]["total_traps"] += 1
            scores["v4"]["total_traps"] += 1
            
            # Vérification V1 (Seuil Cosinus)
            max_score_v1 = max([d['score'] for d in docs_v1]) if docs_v1 else 0
            if not docs_v1 or max_score_v1 < THRESHOLD_SIMPLE:
                scores["v1"]["traps_caught"] += 1
                
            # Vérification V4 (Seuil Rerank)
            max_score_v4 = max([d['score'] for d in docs_v4]) if docs_v4 else -10
            if not docs_v4 or max_score_v4 < THRESHOLD_STRICT:
                scores["v4"]["traps_caught"] += 1

        else:
            # LOGIQUE STANDARD : Succès si l'ID attendu est trouvé
            scores["v1"]["total_scorable"] += 1
            scores["v4"]["total_scorable"] += 1

            if calculate_success(docs_v1, expected_ids):
                scores["v1"]["correct"] += 1
            
            if calculate_success(docs_v4, expected_ids):
                scores["v4"]["correct"] += 1


=== BENCHMARK : V1 (Simple) vs V4 (Hybride) ===
[INFO] 6 questions chargées.
--- Chargement des ressources RAG ---

--- Lancement des tests (Comparaison V1 vs V4) ---


100%|██████████| 6/6 [00:00<00:00, 20.65it/s]


## 8. Rapport Final et Conclusion

Le benchmark est terminé. Voici les résultats définitifs comparant l'approche naïve (V1) et notre architecture avancée (V4).

In [22]:
# 7. RAPPORT FINAL

# Calcul des pourcentages
def get_acc(d):
    retrieval = (d["correct"] / d["total_scorable"] * 100) if d["total_scorable"] > 0 else 0
    traps = (d["traps_caught"] / d["total_traps"] * 100) if d["total_traps"] > 0 else 0
    return retrieval, traps

acc_v1, trap_v1 = get_acc(scores["v1"])
acc_v4, trap_v4 = get_acc(scores["v4"])

print("\n" + "="*60)
print(f"       📊 RAPPORT COMPARATIF (V1 vs V4)       ")
print("="*60)

# Création d'un petit DataFrame pour l'affichage propre
summary_df = pd.DataFrame({
    "Métrique": ["Précision Recherche (Trouve le doc)", "Filtrage Pièges (Anti-Hallucination)"],
    "V1 (Simple FAISS)": [f"{acc_v1:.1f}%", f"{trap_v1:.1f}%"],
    "V4 (Hybride Ultime)": [f"{acc_v4:.1f}%", f"{trap_v4:.1f}%"],
    "Amélioration": [f"+{acc_v4 - acc_v1:.1f}%", f"+{trap_v4 - trap_v1:.1f}%"]
})

# Affichage
print(summary_df.to_string(index=False))
print("\n" + "="*60)


       📊 RAPPORT COMPARATIF (V1 vs V4)       
                            Métrique V1 (Simple FAISS) V4 (Hybride Ultime) Amélioration
 Précision Recherche (Trouve le doc)             40.0%               60.0%       +20.0%
Filtrage Pièges (Anti-Hallucination)            100.0%              100.0%        +0.0%

